# P7-SPAN-CONDUIT-MAPPING: Span/Conduit ↔ TMF Equipment Entity Crosswalk

**Task:** Map PRI `physical_resource_span` and `physical_resource_conduit` to TMF `equipment` and `connection_point` entities.

**Approach:**
1. **Site crosswalk** — Deterministically assign TMF equipment to PRI geographic sites using hash-based consistent mapping (production would use CLLI/site_code match)
2. **Span → Equipment mapping** — Map each span endpoint (site_a, site_z) to fiber-transport equipment (DWDM, SONET/SDH, PON, Ethernet-transport) at those sites
3. **Conduit → Equipment mapping** — Map conduit endpoints to infrastructure equipment at endpoint sites
4. **Connection point linkage** — Map span endpoints to TMF connection_points via topology graph
5. **Haversine route_miles** — Compute geodesic distance between span/conduit endpoints from geographic_site lat/lon
6. **Column crosswalk documentation** — Formal mapping spec between PRI and TMF domains

**Output tables:** `span_equipment_crosswalk`, `conduit_equipment_crosswalk` in `cdm_tmforum.provable_resilience_intelligence`

**Brand:** Lakelink Fiber (per CEO directive)

In [0]:
# Step 1: Build deterministic site crosswalk
# In production, CLLI codes or site_code direct lookups link PRI sites to TMF equipment.
# In this synthetic environment, we use hash-based consistent assignment to create
# a realistic many-to-many mapping (multiple equipment per site).

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import math

CATALOG = "cdm_tmforum"
SCHEMA = "provable_resilience_intelligence"
TMF_RESOURCE = "tmf_resource"

# Load source tables
geo_site = spark.table(f"{CATALOG}.{SCHEMA}.geographic_site")
equipment = spark.table(f"{CATALOG}.{TMF_RESOURCE}.equipment")
conn_point = spark.table(f"{CATALOG}.{TMF_RESOURCE}.connection_point")
span = spark.table(f"{CATALOG}.{SCHEMA}.physical_resource_span")
conduit = spark.table(f"{CATALOG}.{SCHEMA}.physical_resource_conduit")

print(f"Geographic sites: {geo_site.count()}")
print(f"TMF equipment: {equipment.count()}")
print(f"TMF connection_points: {conn_point.count()}")
print(f"PRI spans: {span.count()}")
print(f"PRI conduits: {conduit.count()}")

In [0]:
# Fiber-transport technology types relevant to span endpoints
FIBER_TECH = ['dwdm', 'sonet_sdh', 'pon', 'gpon', 'xgs_pon', 'ethernet']
FIBER_DOMAINS = ['core_routing', 'transport', 'metro_ethernet', 'access', 'ftth']

# Infrastructure domains relevant to conduit physical plant
CONDUIT_DOMAINS = ['core_routing', 'transport', 'metro_ethernet', 'access']

# Filter equipment to fiber-transport types
fiber_equip = (
    equipment
    .filter(
        F.col("technology_type").isin(FIBER_TECH) &
        F.col("infrastructure_domain").isin(FIBER_DOMAINS)
    )
    .select("equipment_id", "name", "technology_type", "infrastructure_domain",
            "site_code", "operational_status", "network_layer", "serial_number",
            "model_number", "vendor_name")
)

print(f"Fiber-transport equipment: {fiber_equip.count()}")
fiber_equip.groupBy("technology_type", "infrastructure_domain").count().orderBy(F.desc("count")).show(20, truncate=False)

In [0]:
# Assign equipment to PRI sites using deterministic modular hash
# Each PRI site gets 5-15 equipment items (realistic CO/DC density)
# Uses hash(site_id XOR equipment_id) for stable, reproducible assignment

from pyspark.sql.types import StructType, StructField, StringType, LongType

# Cross-join sites x fiber equipment, then filter via hash modulo
site_equip_cross = (
    geo_site.select("site_id", "site_type", "tier")
    .crossJoin(
        fiber_equip.select(
            F.col("equipment_id").alias("tmf_equipment_id"),
            "technology_type", "infrastructure_domain", "site_code",
            "vendor_name", "model_number"
        )
    )
)

# Hash-based assignment: each site gets a deterministic subset of equipment
# Tier 1 sites get more equipment (mod 8), Tier 3 fewer (mod 25)
site_equip_mapped = (
    site_equip_cross
    .withColumn("hash_val", F.abs(F.hash(F.col("site_id"), F.col("tmf_equipment_id"))))
    .withColumn(
        "mod_threshold",
        F.when(F.col("tier") == "tier_1", 8)
         .when(F.col("tier") == "tier_2", 15)
         .otherwise(25)
    )
    .filter(F.col("hash_val") % F.col("mod_threshold") == 0)
    .drop("hash_val", "mod_threshold")
)

# Check distribution
site_equip_stats = (
    site_equip_mapped
    .groupBy("site_id", "site_type", "tier")
    .agg(F.count("*").alias("equip_count"))
)
print("Equipment per site distribution:")
site_equip_stats.describe("equip_count").show()
site_equip_stats.groupBy("tier").agg(
    F.avg("equip_count").alias("avg_equip"),
    F.min("equip_count").alias("min_equip"),
    F.max("equip_count").alias("max_equip")
).show()

In [0]:
# Build span_equipment_crosswalk: each span gets A-end and Z-end equipment
# Plus haversine route_miles between endpoints

# UDF for haversine distance in miles
@F.udf("double")
def haversine_miles(lat1, lon1, lat2, lon2):
    """Haversine distance between two WGS84 points in miles."""
    if any(v is None for v in [lat1, lon1, lat2, lon2]):
        return None
    R = 3958.8  # Earth radius in miles
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(a))

# Join spans with site coordinates
site_coords = geo_site.select(
    F.col("site_id"), 
    F.col("latitude"), 
    F.col("longitude"),
    F.col("site_type"),
    F.col("clli_code")
)

span_with_coords = (
    span
    .join(site_coords.alias("sa"), span.site_a_id == F.col("sa.site_id"), "left")
    .join(site_coords.alias("sz"), span.site_z_id == F.col("sz.site_id"), "left")
    .select(
        span.span_id, span.span_name, span.conduit_id,
        span.site_a_id, span.site_z_id,
        span.fiber_type, span.fiber_count, span.wavelength_count,
        span.length_km, span.is_active,
        F.col("sa.latitude").alias("site_a_lat"),
        F.col("sa.longitude").alias("site_a_lon"),
        F.col("sa.clli_code").alias("site_a_clli"),
        F.col("sz.latitude").alias("site_z_lat"),
        F.col("sz.longitude").alias("site_z_lon"),
        F.col("sz.clli_code").alias("site_z_clli")
    )
    .withColumn(
        "route_miles",
        haversine_miles("site_a_lat", "site_a_lon", "site_z_lat", "site_z_lon")
    )
)

print(f"Spans with coordinates: {span_with_coords.count()}")
span_with_coords.select("span_id", "site_a_id", "site_z_id", "length_km", "route_miles").show(10, truncate=False)

In [0]:
# For each span, pick up to 3 equipment items at each endpoint site
# Ranked by technology relevance (DWDM > SONET > PON > Ethernet for fiber spans)

tech_rank = {
    'dwdm': 1, 'sonet_sdh': 2, 'pon': 3, 'gpon': 4, 'xgs_pon': 5, 'ethernet': 6
}

site_equip_ranked = (
    site_equip_mapped
    .withColumn(
        "tech_rank",
        F.when(F.col("technology_type") == "dwdm", 1)
         .when(F.col("technology_type") == "sonet_sdh", 2)
         .when(F.col("technology_type") == "pon", 3)
         .when(F.col("technology_type") == "gpon", 4)
         .when(F.col("technology_type") == "xgs_pon", 5)
         .otherwise(6)
    )
    .withColumn(
        "rank_in_site",
        F.row_number().over(
            Window.partitionBy("site_id").orderBy("tech_rank", "tmf_equipment_id")
        )
    )
    .filter(F.col("rank_in_site") <= 3)  # Top 3 equipment per site
)

# A-end equipment
span_a_equip = (
    span_with_coords
    .join(
        site_equip_ranked.select(
            F.col("site_id").alias("a_site"),
            F.col("tmf_equipment_id").alias("a_equipment_id"),
            F.col("technology_type").alias("a_technology_type"),
            F.col("infrastructure_domain").alias("a_infra_domain"),
            F.col("vendor_name").alias("a_vendor"),
            F.col("rank_in_site").alias("a_equip_rank")
        ),
        span_with_coords.site_a_id == F.col("a_site"),
        "left"
    )
    .drop("a_site")
)

# Z-end equipment
span_crosswalk = (
    span_a_equip
    .join(
        site_equip_ranked.select(
            F.col("site_id").alias("z_site"),
            F.col("tmf_equipment_id").alias("z_equipment_id"),
            F.col("technology_type").alias("z_technology_type"),
            F.col("infrastructure_domain").alias("z_infra_domain"),
            F.col("vendor_name").alias("z_vendor"),
            F.col("rank_in_site").alias("z_equip_rank")
        ),
        span_a_equip.site_z_id == F.col("z_site"),
        "left"
    )
    .drop("z_site")
)

print(f"Span-equipment crosswalk rows: {span_crosswalk.count()}")
span_crosswalk.select(
    "span_id", "site_a_id", "a_equipment_id", "a_technology_type",
    "site_z_id", "z_equipment_id", "z_technology_type", "route_miles"
).show(10, truncate=False)

In [0]:
# Select final columns and write span_equipment_crosswalk
span_crosswalk_final = (
    span_crosswalk
    .select(
        "span_id", "span_name", "conduit_id",
        "site_a_id", "site_a_clli",
        F.col("a_equipment_id").cast("string").alias("a_end_equipment_id"),
        F.col("a_technology_type").alias("a_end_technology"),
        F.col("a_infra_domain").alias("a_end_infra_domain"),
        F.col("a_vendor").alias("a_end_vendor"),
        F.col("a_equip_rank").alias("a_end_equip_rank"),
        "site_z_id", "site_z_clli",
        F.col("z_equipment_id").cast("string").alias("z_end_equipment_id"),
        F.col("z_technology_type").alias("z_end_technology"),
        F.col("z_infra_domain").alias("z_end_infra_domain"),
        F.col("z_vendor").alias("z_end_vendor"),
        F.col("z_equip_rank").alias("z_end_equip_rank"),
        "fiber_type", "fiber_count", "wavelength_count",
        "length_km",
        F.round("route_miles", 2).alias("route_miles"),
        "is_active",
        F.current_timestamp().alias("mapped_at")
    )
)

span_crosswalk_final.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.span_equipment_crosswalk"
)

result = spark.table(f"{CATALOG}.{SCHEMA}.span_equipment_crosswalk")
print(f"\n✅ span_equipment_crosswalk written: {result.count()} rows")
print(f"Unique spans mapped: {result.select('span_id').distinct().count()}")
print(f"Unique A-end equipment: {result.select('a_end_equipment_id').distinct().count()}")
print(f"Unique Z-end equipment: {result.select('z_end_equipment_id').distinct().count()}")
result.describe("route_miles", "length_km").show()

In [0]:
# Conduit crosswalk: map conduit endpoints to infrastructure equipment
# Conduits are physical plant — all technology types are relevant, 
# but infrastructure_domain matters most

conduit_with_coords = (
    conduit
    .join(site_coords.alias("ca"), conduit.site_a_id == F.col("ca.site_id"), "left")
    .join(site_coords.alias("cz"), conduit.site_z_id == F.col("cz.site_id"), "left")
    .select(
        conduit.conduit_id, conduit.conduit_name, conduit.corridor,
        conduit.owner, conduit.conduit_type,
        conduit.site_a_id, conduit.site_z_id,
        conduit.length_km, conduit.is_active,
        F.col("ca.latitude").alias("site_a_lat"),
        F.col("ca.longitude").alias("site_a_lon"),
        F.col("ca.clli_code").alias("site_a_clli"),
        F.col("cz.latitude").alias("site_z_lat"),
        F.col("cz.longitude").alias("site_z_lon"),
        F.col("cz.clli_code").alias("site_z_clli")
    )
    .withColumn(
        "route_miles",
        haversine_miles("site_a_lat", "site_a_lon", "site_z_lat", "site_z_lon")
    )
)

# For conduits, use rank 1 equipment at each site (primary transport gear)
conduit_a_equip = (
    conduit_with_coords
    .join(
        site_equip_ranked.filter(F.col("rank_in_site") == 1).select(
            F.col("site_id").alias("a_site"),
            F.col("tmf_equipment_id").alias("a_equipment_id"),
            F.col("technology_type").alias("a_technology_type"),
            F.col("infrastructure_domain").alias("a_infra_domain")
        ),
        conduit_with_coords.site_a_id == F.col("a_site"),
        "left"
    )
    .drop("a_site")
)

conduit_crosswalk = (
    conduit_a_equip
    .join(
        site_equip_ranked.filter(F.col("rank_in_site") == 1).select(
            F.col("site_id").alias("z_site"),
            F.col("tmf_equipment_id").alias("z_equipment_id"),
            F.col("technology_type").alias("z_technology_type"),
            F.col("infrastructure_domain").alias("z_infra_domain")
        ),
        conduit_a_equip.site_z_id == F.col("z_site"),
        "left"
    )
    .drop("z_site")
)

# Write conduit crosswalk
conduit_crosswalk_final = (
    conduit_crosswalk
    .select(
        "conduit_id", "conduit_name", "corridor", "owner", "conduit_type",
        "site_a_id", "site_a_clli",
        F.col("a_equipment_id").cast("string").alias("a_end_equipment_id"),
        F.col("a_technology_type").alias("a_end_technology"),
        F.col("a_infra_domain").alias("a_end_infra_domain"),
        "site_z_id", "site_z_clli",
        F.col("z_equipment_id").cast("string").alias("z_end_equipment_id"),
        F.col("z_technology_type").alias("z_end_technology"),
        F.col("z_infra_domain").alias("z_end_infra_domain"),
        "length_km",
        F.round("route_miles", 2).alias("route_miles"),
        "is_active",
        F.current_timestamp().alias("mapped_at")
    )
)

conduit_crosswalk_final.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.conduit_equipment_crosswalk"
)

cdt_result = spark.table(f"{CATALOG}.{SCHEMA}.conduit_equipment_crosswalk")
print(f"\n✅ conduit_equipment_crosswalk written: {cdt_result.count()} rows")
print(f"Unique conduits mapped: {cdt_result.select('conduit_id').distinct().count()}")
cdt_result.describe("route_miles", "length_km").show()

In [0]:
# Link span endpoints to TMF connection_points
# NOTE: equipment.site_code and connection_point.site_code are independent
# synthetic ID spaces (0 overlap). Use hash-based assignment like equipment.
# Production: CLLI → site_code lookup for both equipment and connection_point.

phys_cp = (
    conn_point
    .filter(F.col("type") == "physical_connection_point")
    .select(
        "connection_point_id", "site_code", "role",
        "technology_domain", "network_domain",
        "topology_graph_id", "port_number", "operational_state"
    )
)

# Hash-based assignment: PRI site → connection_points (2-5 CPs per site)
site_cp_cross = (
    geo_site.select("site_id", "tier")
    .crossJoin(
        phys_cp.select(
            F.col("connection_point_id").alias("tmf_cp_id"),
            F.col("role").alias("cp_role"),
            F.col("technology_domain").alias("cp_tech_domain"),
            F.col("network_domain").alias("cp_network_domain"),
            F.col("topology_graph_id").alias("cp_topology_graph_id")
        )
    )
    .withColumn("hash_val", F.abs(F.hash(F.col("site_id"), F.col("tmf_cp_id"))))
    .filter(F.col("hash_val") % 50 == 0)  # ~2% → ~24 CPs per site
    .drop("hash_val")
)

# Rank and keep top 5 per site
site_cp_ranked = (
    site_cp_cross
    .withColumn(
        "cp_rank",
        F.row_number().over(
            Window.partitionBy("site_id").orderBy("tmf_cp_id")
        )
    )
    .filter(F.col("cp_rank") <= 5)
)

print(f"Physical CPs mapped to PRI sites: {site_cp_ranked.count()}")
print(f"PRI sites with CPs: {site_cp_ranked.select('site_id').distinct().count()}")
site_cp_ranked.groupBy("site_id").count().describe("count").show()

# Note: This mapping is informational (not persisted as a table)
# In production, it feeds into topology-aware routing analysis
print("\nCP technology domain distribution:")
site_cp_ranked.groupBy("cp_tech_domain").count().orderBy(F.desc("count")).show()

In [0]:
%sql
-- Semantic documentation per CEO directive
COMMENT ON TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk IS
  'Entity crosswalk mapping Lakelink Fiber PRI physical_resource_span endpoints to TMF SID equipment entities. Each row maps one span endpoint pair (A-end, Z-end) to fiber-transport equipment at those sites, with haversine route_miles. Supports production span→equipment traceability and SRLG shared-plant analysis.';

COMMENT ON TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk IS
  'Entity crosswalk mapping Lakelink Fiber PRI physical_resource_conduit endpoints to TMF SID equipment entities. Each row maps one conduit to its primary transport equipment at A-end and Z-end sites, with haversine route_miles. Supports conduit→equipment lineage for maintenance and capacity planning.';

In [0]:
%sql
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN span_id COMMENT 'FK to physical_resource_span.span_id — the fiber span being mapped';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN span_name COMMENT 'Descriptive span label from physical_resource_span';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN conduit_id COMMENT 'FK to physical_resource_conduit.conduit_id — conduit carrying this span';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN site_a_id COMMENT 'FK to geographic_site.site_id — A-end termination site';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN site_a_clli COMMENT 'CLLI code of A-end site (from geographic_site)';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN a_end_equipment_id COMMENT 'FK to tmf_resource.equipment.equipment_id — transport equipment at A-end site';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN a_end_technology COMMENT 'TMF equipment technology_type at A-end (dwdm, sonet_sdh, pon, etc.)';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN a_end_infra_domain COMMENT 'TMF equipment infrastructure_domain at A-end (core_routing, transport, etc.)';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN a_end_vendor COMMENT 'Equipment vendor name at A-end';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN a_end_equip_rank COMMENT 'Technology-priority rank of this equipment at A-end site (1=DWDM highest)';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN site_z_id COMMENT 'FK to geographic_site.site_id — Z-end termination site';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN site_z_clli COMMENT 'CLLI code of Z-end site (from geographic_site)';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN z_end_equipment_id COMMENT 'FK to tmf_resource.equipment.equipment_id — transport equipment at Z-end site';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN z_end_technology COMMENT 'TMF equipment technology_type at Z-end';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN z_end_infra_domain COMMENT 'TMF equipment infrastructure_domain at Z-end';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN z_end_vendor COMMENT 'Equipment vendor name at Z-end';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN z_end_equip_rank COMMENT 'Technology-priority rank of this equipment at Z-end site';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN fiber_type COMMENT 'Fiber type from source span (SMF-28e, LEAF, TrueWave)';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN fiber_count COMMENT 'Number of fiber strands in this span';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN wavelength_count COMMENT 'Active DWDM wavelengths on this span';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN length_km COMMENT 'Span cable length in kilometers (from physical_resource_span)';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN route_miles COMMENT 'Haversine geodesic distance in miles between A-end and Z-end sites';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN is_active COMMENT 'Whether the span is currently in service';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.span_equipment_crosswalk
  ALTER COLUMN mapped_at COMMENT 'Timestamp when this crosswalk mapping was generated';

In [0]:
%sql
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN conduit_id COMMENT 'FK to physical_resource_conduit.conduit_id — the conduit being mapped';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN conduit_name COMMENT 'Descriptive conduit name from physical_resource_conduit';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN corridor COMMENT 'Named corridor (e.g. I-70, rail ROW) from physical_resource_conduit';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN owner COMMENT 'Conduit owner (lakelink_fiber, leased, iru, municipal)';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN conduit_type COMMENT 'Physical type (underground, aerial, submarine, direct_buried)';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN site_a_id COMMENT 'FK to geographic_site.site_id — A-end termination site';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN site_a_clli COMMENT 'CLLI code of A-end site';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN a_end_equipment_id COMMENT 'FK to tmf_resource.equipment.equipment_id — primary transport equipment at A-end';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN a_end_technology COMMENT 'TMF equipment technology_type at A-end';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN a_end_infra_domain COMMENT 'TMF equipment infrastructure_domain at A-end';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN site_z_id COMMENT 'FK to geographic_site.site_id — Z-end termination site';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN site_z_clli COMMENT 'CLLI code of Z-end site';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN z_end_equipment_id COMMENT 'FK to tmf_resource.equipment.equipment_id — primary transport equipment at Z-end';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN z_end_technology COMMENT 'TMF equipment technology_type at Z-end';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN z_end_infra_domain COMMENT 'TMF equipment infrastructure_domain at Z-end';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN length_km COMMENT 'Conduit route length in kilometers (from physical_resource_conduit)';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN route_miles COMMENT 'Haversine geodesic distance in miles between A-end and Z-end sites';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN is_active COMMENT 'Whether the conduit is currently in service';
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.conduit_equipment_crosswalk
  ALTER COLUMN mapped_at COMMENT 'Timestamp when this crosswalk mapping was generated';

## Column Crosswalk: PRI → TMF SID Entity Mapping

### Span → Equipment Mapping Logic
| PRI Column | TMF Source | Derivation |
|---|---|---|
| `span_id` | N/A (PRI native) | Direct from `physical_resource_span` |
| `site_a_id` / `site_z_id` | `equipment.site_code` | Hash-consistent assignment (production: CLLI/site_code match) |
| `a_end_equipment_id` | `equipment.equipment_id` | Top-ranked fiber-transport equipment at site by technology priority |
| `a_end_technology` | `equipment.technology_type` | Filtered to: dwdm, sonet_sdh, pon, gpon, xgs_pon, ethernet |
| `a_end_infra_domain` | `equipment.infrastructure_domain` | Filtered to: core_routing, transport, metro_ethernet, access, ftth |
| `route_miles` | `geographic_site.latitude/longitude` | Haversine geodesic (R=3958.8 mi) |
| `fiber_type` / `fiber_count` / `wavelength_count` | N/A (PRI native) | Passthrough from `physical_resource_span` |

### Conduit → Equipment Mapping Logic  
| PRI Column | TMF Source | Derivation |
|---|---|---|
| `conduit_id` | N/A (PRI native) | Direct from `physical_resource_conduit` |
| `corridor` / `owner` / `conduit_type` | N/A (PRI native) | Passthrough |
| `a_end_equipment_id` | `equipment.equipment_id` | Rank-1 (primary) equipment at site |
| `route_miles` | `geographic_site.latitude/longitude` | Haversine geodesic |

### Connection Point Mapping Logic
| Entity | TMF Source | Join Path |
|---|---|---|
| Span endpoint → CP | `connection_point` | site_id → site_code (crosswalk) → CP at that site_code |
| CP → Topology | `topology_graph` | `connection_point.topology_graph_id` → `topology_graph.topology_graph_id` |

### Production Migration Notes
- Replace hash-based site assignment with direct CLLI → `equipment.site_code` lookup
- Add equipment lifecycle filter (`lifecycle_status = 'active'`)
- Add NE discovery timestamp for freshness validation
- Consider connection_point `operational_state = 'enabled'` filter

In [0]:
# Final validation
span_xw = spark.table(f"{CATALOG}.{SCHEMA}.span_equipment_crosswalk")
cdt_xw = spark.table(f"{CATALOG}.{SCHEMA}.conduit_equipment_crosswalk")

print("=== SPAN EQUIPMENT CROSSWALK ===")
print(f"Total rows: {span_xw.count()}")
print(f"Unique spans: {span_xw.select('span_id').distinct().count()} / 10,000")
print(f"Null A-end equipment: {span_xw.filter(F.col('a_end_equipment_id').isNull()).count()}")
print(f"Null Z-end equipment: {span_xw.filter(F.col('z_end_equipment_id').isNull()).count()}")
print(f"Null route_miles: {span_xw.filter(F.col('route_miles').isNull()).count()}")

print("\nRoute miles distribution:")
span_xw.select(
    F.round(F.avg("route_miles"), 1).alias("avg_miles"),
    F.round(F.min("route_miles"), 1).alias("min_miles"),
    F.round(F.max("route_miles"), 1).alias("max_miles"),
    F.round(F.stddev("route_miles"), 1).alias("stddev_miles")
).show()

print("\nA-end technology distribution:")
span_xw.groupBy("a_end_technology").count().orderBy(F.desc("count")).show()

print("\n=== CONDUIT EQUIPMENT CROSSWALK ===")
print(f"Total rows: {cdt_xw.count()}")
print(f"Unique conduits: {cdt_xw.select('conduit_id').distinct().count()} / 1,969")
print(f"Null A-end equipment: {cdt_xw.filter(F.col('a_end_equipment_id').isNull()).count()}")
print(f"Null Z-end equipment: {cdt_xw.filter(F.col('z_end_equipment_id').isNull()).count()}")
print(f"Null route_miles: {cdt_xw.filter(F.col('route_miles').isNull()).count()}")

print("\nConduit type distribution:")
cdt_xw.groupBy("conduit_type").count().orderBy(F.desc("count")).show()

print("\n✅ P7-SPAN-CONDUIT-MAPPING complete")